# Stokes architecture recovery restore

Two epochs of eight training meshes per arm at production width. The exact interruption boundary is epoch one. Deterministic replay requires bitwise agreement.


In [ ]:
from pathlib import Path
import re
import subprocess
import tempfile

REPO_URL = "https://github.com/AdebanjiAdelowo/physicsnemo-stokes-finetuning.git"
REF = "3fdc44777a6bd9250825dcd11af08cb1df0e7088"
assert re.fullmatch(r"[0-9a-f]{40}", REF)
REPO = Path(tempfile.mkdtemp(prefix="stokes-arch-")) / "physicsnemo-stokes-finetuning"
OUTPUT = Path("/kaggle/working")

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

def tree_is_clean():
    return not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()

subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", REF], cwd=REPO, check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert head == REF and tree_is_clean(), f"expected clean checkout of {REF}, got {head}"
print("Probe commit", head)
subprocess.run(["nvidia-smi"], check=True)

import hashlib
EXPORT_SHA = "__VERIFIED_EXPORT_SHA256__"
assert re.fullmatch(r"[0-9a-f]{64}", EXPORT_SHA), "bind the independently downloaded export manifest first"
manifests = sorted(Path("/kaggle/input").rglob("recovery-export.json"))
assert len(manifests) == 1, f"expected one attached export manifest, found {manifests}"
assert hashlib.sha256(manifests[0].read_bytes()).hexdigest() == EXPORT_SHA, "export manifest hash differs"
SOURCE = manifests[0].parent


In [ ]:
sh("pip install -q -r requirements.txt", cwd=REPO)
sh("FORCE_ONLY_CPU=1 pip install -q --no-build-isolation --no-binary torch-scatter torch-scatter==2.1.2", cwd=REPO)
# The Kaggle image preinstalls RAPIDS (cuml, cudf, cupy, rmm, ...) and numba. PhysicsNeMo imports cuml
# whenever it is installed; cudf then imports numba, which rejects the NumPy that requirements.txt
# installs (first launch, 2026-11-07: "Numba needs NumPy 2.2 or less. Got NumPy 2.5"). The validated
# environment (requirements.lock.txt) contains none of these packages, so they are removed from this
# session instead of downgrading NumPy.
RAPIDS_IMPORTS = ("cuml", "cudf", "dask_cudf", "cugraph", "pylibcugraph", "pylibcugraphops", "nx_cugraph", "cuspatial",
                  "cuvs", "cuxfilter", "cucim", "rmm", "pylibraft", "raft_dask", "cupy", "numba", "numba_cuda")
sh("python -c \"from importlib.metadata import packages_distributions as p; m = p(); "
   f"print(' '.join(sorted({{d for name in {RAPIDS_IMPORTS!r} for d in m.get(name, [])}})))\" > /tmp/rapids_dists.txt")
dists = open("/tmp/rapids_dists.txt").read().split()
print("removing from this session:", dists)
if dists:
    sh("pip uninstall -y -q " + " ".join(dists))
sh('python -c "import torch, physicsnemo, torch_geometric, torch_scatter; assert torch.cuda.is_available(); '
   "print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| physicsnemo', physicsnemo.__version__, "
   "'| torch_geometric', torch_geometric.__version__, '| torch_scatter', torch_scatter.__version__, '|', torch.cuda.get_device_name(0))\"", cwd=REPO)
# every project module imports on this environment before any data or GPU time is spent
sh('PYTHONPATH=src python -c "import numpy, stokes_ft.data, stokes_ft.engine, stokes_ft.model, stokes_ft.study, '
   "stokes_ft.physics, stokes_ft.weak, stokes_ft.variants, stokes_ft.fem; print('project imports OK | numpy', numpy.__version__)\"", cwd=REPO)

In [ ]:
import shlex
sh("python scripts/prepare_data.py", cwd=REPO)
assert tree_is_clean()
sh(f"CUDA_VISIBLE_DEVICES=0 python scripts/full_arch_cross_session.py restore --output {OUTPUT} --source {shlex.quote(str(SOURCE))}", cwd=REPO)
assert (OUTPUT / "recovery-restore.json").is_file() and tree_is_clean()
